# **Exercise 5: Automated GEOGLOWS Streamflow Forecast Ingestion with HydroServer ETL Tasks**

### **Overview**

This exercise demonstrates how to configure HydroServer's **native ETL orchestration system** to automatically retrieve streamflow forecasts from the [**GEOGLOWS River Forecast System**](https://geoglows.ecmwf.int/?utm_source) and load them into a HydroServer Datastream.

The example uses **GEOGLOWS V2 River ID `160268504`**, corresponding to a modeled river reach on the **Nzoia River near its outlet to Lake Victoria, Kenya**.

The workflow will:

1. Connect to HydroServer and reuse an existing Workspace.

2. Create the HydroServer resources required for the forecast Datastream.

3. Create a reusable **Data Connection** to the GEOGLOWS API.

4. Create an **ETL Task** that maps `flow_median` to the forecast Datastream.

5. Schedule the ETL Task to run automatically once per day.

6. Optionally trigger the task manually to test the configuration.

### **Prerequisites**

This exercise demonstrates functionality that requires a **deployed HydroServer instance with the ETL orchestration system installed and running**.

The public **HydroServer Playground** does not currently support running automated ETL Tasks. Therefore, we will **not run this exercise during the session**. Instead, we will walk through the main steps together, and the notebook is provided for **future reference and practice** with a deployed HydroServer instance.

> We will **not run this exercise during the session**, but we will **walk through the main steps together**. The complete exercise is provided for **future reference and practice**.

### **Software Requirements**

This notebook uses **Python** and the **`hydroserverpy`** Python package to interact with HydroServer and configure automated ETL Tasks.

More detailed examples and documentation are available at:

- [HydroServer Python Client Documentation](https://hydroserver2.github.io/hydroserver/user-guides/how-to/using-the-python-client.html)
- [HydroServer](https://www.hydroserver.org)
- [GEOGLOWS](https://geoglows.ecmwf.int/)


## 1. **Getting Started**

---

### **Install hydroserverpy**

For this workshop, we will use Google Colab to run the exercises. Before starting, run the code cell below to install the required version of the hydroserverpy package. The current version of hydroserverpy used for this training is [1.11.3.](https://pypi.org/project/hydroserverpy/)

In [4]:
%pip install -q hydroserverpy==1.11.3

### **Import the Required Python Packages**

First, import the Python packages needed to connect to HydroServer and configure the automated ETL workflow.

- **`hydroserverpy`** – Connects to HydroServer and allows us to create and manage HydroServer resources and ETL Tasks programmatically.

- **`getpass`** – Allows you to enter your HydroServer password securely without displaying it on the screen.

- **`time`** – Provides time-related functions, such as adding a short delay while waiting for an ETL Task to run.

In [5]:
# Securely enter the HydroServer password without displaying it
from getpass import getpass

# Provides time-related functions, such as adding delays
import time

# Connect to HydroServer and manage resources and ETL Tasks programmatically
from hydroserverpy import HydroServer

### **Set the Initial Parameters to Connect to HydroServer**

The first step in interacting with a HydroServer instance is to establish a connection to it. For this example, we will use your username (email) and password because we will create the workspace programmatically.

When you run the code, you will be prompted to enter your password.

**IMPORTANT: In the following code, change the email to match the HydroServer user account you created.**


In [6]:
# Set initial parameters to connect to HydroServer
hydroserver_host = 'https://playground.hydroserver.org'

# Change the email and password below to your HydroServer username and password
hydroserver_email = 'your email' #'user@youremail.com'
hydroserver_password = getpass('Enter your HydroServer password: ') #getpass('Enter your HydroServer password: ')

Enter your HydroServer password: ··········


### **Initialize HydroServer Connection**

Initialize the connection to HydroServer with the connection information specified above.

In [7]:
# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')


Successfully connected to HydroServer!


### **Get Your Workspace ID**

You need your **Workspace ID** to specify the workspace where the monitoring site and datastream will be created.

In [32]:
# Enter the name of the workspace you created in Exercise 1
workspace_name = "Kenya Training 2026 - your name"

workspaces = hs.workspaces.list(
    is_associated=True
).fetch_all()

workspace_id = next(
    workspace.uid
    for workspace in workspaces.items
    if workspace.name == workspace_name
)

print("Selected workspace:", workspace_name)
print("Workspace ID:", workspace_id)

Selected workspace: Kenya Training 2026 - your name
Workspace ID: 01a07c47-90d4-73c3-911b-d51d0311b2fb


## **2. Create the Monitoring Site**

Next, create a **monitoring site (Thing)** in HydroServer for the forecast location.

We will use the **Nzoia River near its outlet to Lake Victoria**, corresponding to **GEOGLOWS River ID `160268504`**. The streamflow forecasts retrieved from GEOGLOWS will be associated with this location in HydroServer.

The Monitoring Site ID is saved because it will be needed when creating the forecast Datastream.

In [33]:
# Create the monitoring site for the Nzoia River near its outlet to Lake Victoria
nzoia_station = hs.things.create(
    workspace=workspace_id,
    name="Nzoia River near Lake Victoria",
    description=(
        "Location on the Nzoia River near its outlet to Lake Victoria, Kenya. "
        "The corresponding GEOGLOWS V2 modeled river reach is 160268504."
    ),
    sampling_feature_type="Site",
    sampling_feature_code="NZOIA_LAKE_VICTORIA",
    site_type="Stream",
    latitude = 0.05693,
    longitude = 33.95306,
    country="KE",
    is_private=False
)

thing_id = nzoia_station.uid

print("Created monitoring site:")
print(f"{nzoia_station.name}: {thing_id}")

Created monitoring site:
Nzoia River near Lake Victoria: 01a08717-417a-778e-873a-455b51a1c46e


## **3. Create the Datastream**

---

In the following sections, we will create the necessary metadata to load data for a time series of observations recorded at a monitoring site. You can create this metadata using the web user interface of the Data Management App, or you can do it using code, which we are demonstrating here.

HydroServer uses a modified version of the OGC SensorThings API data model for storing time series data and their associated metadata. HydroServer's data model includes the following important entities that we need to create before loading data:

* **Observation Method**: The instrument or method used to measure or create the Observation values.
* **Observed Property**: The variable that is measured (e.g., discharge, water temperature, etc.).
* **Units of Measure**: The units of measure associated with the Observation values (e.g, cubic meters per second).
* **Processing Level**: The degree of processing that has been applied to the Observation values (e.g., "Raw" or "Quality Controlled").
* **Datastream**: A description of the time series that includes all of these attributes.

Once all of these metadata tables have been populated, the time series of data values can be loaded to the **Observations** table in the database.

**NOTE**: To create objects in HydroServer, you will have to pass their required and optional metadata elements. For more information about HydroServer's data model and a data dictionary that describes each of the entities, see HydroServer's documentation at https://www.hydroserver.org.

### **Create an Observation Method (Sensor)**

Next, create a **Sensor** in HydroServer to describe the source of the forecast data.

In this exercise, the Sensor represents the **GEOGLOWS River Forecast System (RFS) V2**, a global hydrologic modeling system that provides modeled streamflow forecasts for river reaches worldwide.

The forecast data will be retrieved from the **GEOGLOWS Data Service API** and stored in HydroServer.

In [34]:
# Create metadata describing GEOGLOWS RFS V2 as the forecast source/model
geoglows_sensor = hs.sensors.create(
    workspace=workspace_id,
    name="GEOGLOWS River Forecast System (RFS) V2",
    description=(
        "Global modeled streamflow forecasts from the GEOGLOWS River Forecast "
        "System (RFS) V2, accessed through the GEOGLOWS Data Service API. "
        "This sensor represents the model source used to generate streamflow "
        "forecasts for GEOGLOWS river reach 160193736."
    ),
    encoding_type="text/csv",
    method_type="Model Simulation",
    method_code="geoglows-rfs-v2"
)

print("Created sensor/model metadata:")
print(f"{geoglows_sensor.name}: {geoglows_sensor.uid}")

Created sensor/model metadata:
GEOGLOWS River Forecast System (RFS) V2: 01a08717-49af-7a7a-863f-076dc253b68d


### **Create the Observed Property**

Next, create the **Observed Property** that defines the variable represented by the forecast data.

For this exercise, the Observed Property is **Streamflow**, representing the forecasted river discharge provided by GEOGLOWS.

The Observed Property ID will later be associated with the forecast Datastream.

In [35]:
# Create the observed property
streamflow = hs.observedproperties.create(
    workspace=workspace_id,
    name="Streamflow",
    definition="Streamflow",
    description="Forecasted river discharge (streamflow).",
    observed_property_type="Hydrology",
    code="Streamflow"
)

print("Created observed property:")
print(f"{streamflow.name}: {streamflow.uid}")

Created observed property:
Streamflow: 01a08717-4d4b-7ac2-bce1-cd6e7517b5db


### **Create Units of Measure**

Next, create the **Unit** associated with the Streamflow Observed Property.

GEOGLOWS reports streamflow forecasts in **cubic meters per second (m³/s)**. This unit will later be associated with the forecast Datastream.

In [36]:
# GEOGLOWS streamflow is reported in cubic meters per second
streamflow_unit = hs.units.create(
    workspace=workspace_id,
    name="Cubic meter per second",
    symbol="m^3/s",
    definition="Cubic meters per second",
    unit_type="Volumetric Flow Rate"
)

print("Created unit:")
print(f"{streamflow_unit.name}: {streamflow_unit.uid}")

Created unit:
Cubic meter per second: 01a08717-521f-772b-85ee-a0ac6d9a7c98


### **Create a Processing Level**

Next, create a **Processing Level** to indicate that the streamflow values are **forecasted model outputs** rather than observed measurements.

For this exercise, the Processing Level identifies the data as streamflow forecasts produced by **GEOGLOWS RFS V2**. It will later be associated with the forecast Datastream.

In [37]:
# Processing level for forecast/model output
forecast_processing = hs.processinglevels.create(
    workspace=workspace_id,
    code="Forecast",
    definition="Forecast streamflow",
    explanation="Streamflow forecast produced by GEOGLOWS RFS V2."
)

print("Created processing level:")
print(f"{forecast_processing.code}: {forecast_processing.uid}")

Created processing level:
Forecast: 01a08717-5666-7688-bc74-e9c3e4a23c80


### **Create the GEOGLOWS Forecast Datastream**

Next, create the **Datastream** that will store the GEOGLOWS streamflow forecast in HydroServer.

The Datastream connects the **Nzoia River monitoring site** with the GEOGLOWS model, Streamflow Observed Property, unit, and Forecast Processing Level created in the previous steps.

Because GEOGLOWS provides forecasted streamflow at **3-hour intervals**, the Datastream is configured with a 3-hour time spacing and a forecast period extending up to **15 days ahead**.

In [38]:
forecast_datastream = hs.datastreams.create(
    name=f"GEOGLOWS Streamflow Forecast - {nzoia_station.name}",
    description=(
        "Latest GEOGLOWS RFS V2 streamflow forecast for river reach 160193736, "
        "near gauge 1DA02 on the Nzoia River."
    ),
    thing=nzoia_station.uid,
    sensor=geoglows_sensor.uid,
    observed_property=streamflow.uid,
    processing_level=forecast_processing.uid,
    unit=streamflow_unit.uid,
    observation_type="Model Simulation",
    result_type="Timeseries",
    sampled_medium="Surface Water",
    no_data_value=-9999,
    aggregation_statistic="Average",
    time_aggregation_interval=3,
    time_aggregation_interval_unit="hours",
    intended_time_spacing=3,
    intended_time_spacing_unit="hours",
    status="Ongoing",
    is_private=False,
    is_visible=True
)

### **HydroServer-Native ETL Orchestration**

In this version of the exercise, we do **not** manually build an `ETLPipeline` with an Extractor, Transformer, and Loader.

Instead, HydroServer stores the source configuration as a **Data Connection** and the mapping/schedule as an **ETL Task**. When orchestration is enabled, HydroServer's worker executes the task automatically according to its schedule.


### **Create the Data Connection**

A **Data Connection** tells HydroServer where the source data are located and how to read them.

For this exercise, the source is the GEOGLOWS V2 forecast API. The river reach is defined as a `per_task` placeholder, making the same Data Connection reusable for other GEOGLOWS river reaches.

The API returns a CSV file in which the `datetime` column contains forecast timestamps.


In [39]:
# Create a reusable Data Connection to the GEOGLOWS forecast API
geoglows_data_connection = hs.dataconnections.create(

    # Name used to identify this Data Connection in HydroServer
    name="GEOGLOWS Streamflow Forecasts",

    # Workspace where the Data Connection will be created
    workspace=workspace_id,

    # URL used to retrieve the GEOGLOWS forecast
    # {river_id} is a placeholder that will be replaced by the River ID
    # specified later in each ETL Task
    source_url=(
        "https://geoglows.ecmwf.int/api/v2/"
        "forecast/{river_id}?format=csv"
    ),

    # Format of the data returned by the GEOGLOWS API
    payload_type="CSV",

    # Name of the CSV column containing the observation timestamps
    timestamp_key="datetime",

    # Row containing the CSV column names
    header_row=1,

    # Row where the forecast data values begin
    data_start_row=2,

    # Character used to separate values in the CSV file
    delimiter=",",

    # Define the placeholder variables used in the source URL
    placeholder_variables=[
        {
            # Corresponds to {river_id} in the source URL
            "name": "river_id",

            # The River ID will be specified separately for each ETL Task,
            # allowing this Data Connection to be reused for different rivers
            "variable_type": "per_task"
        }
    ],
)

# Display the name and unique ID of the created Data Connection
print(f"{geoglows_data_connection.name}: {geoglows_data_connection.uid}")

GEOGLOWS Streamflow Forecasts: 01a08717-6585-7324-89d0-7e05bd298cea


### **Create and Schedule the GEOGLOWS ETL Task**

Next, create an **ETL Task** that connects the GEOGLOWS Data Connection to the HydroServer forecast Datastream.

The task provides **GEOGLOWS River ID `160268504`**, corresponding to the Nzoia River near its outlet to Lake Victoria, and maps the `flow_median` values from GEOGLOWS to the target HydroServer Datastream.

The task is configured to run **once per day** when HydroServer's ETL orchestration system is enabled.

> **Note:** A HydroServer Datastream can only be mapped to one ETL Task at a time. If this cell has already been run successfully, running it again may return an error indicating that the Datastream is already mapped to another task.

In [40]:
# Create a scheduled ETL Task for the Nzoia River GEOGLOWS forecast
geoglows_etl_task = hs.etltasks.create(

    # Name of the ETL Task
    name="Daily Nzoia GEOGLOWS Streamflow Forecast",

    # Data Connection that defines where and how to retrieve the GEOGLOWS data
    data_connection=geoglows_data_connection,

    # Replace {river_id} in the Data Connection URL with the GEOGLOWS River ID
    task_variables={
        "river_id": "160268504"
    },

    # Map the GEOGLOWS forecast variable to the HydroServer Datastream
    mappings=[
        {
            "source_identifier": "flow_median",
            "target_datastream_id": str(forecast_datastream.uid),
        }
    ],

    # Run the ETL Task once every day
    interval=1,
    interval_period="days",

    # Enable automatic execution of the ETL Task
    enabled=True,
)

# Display information about the created ETL Task
print(f"{geoglows_etl_task.name}: {geoglows_etl_task.uid}")
print(f"Enabled: {geoglows_etl_task.enabled}")
print(f"Next run: {geoglows_etl_task.next_run_at}")

Daily Nzoia GEOGLOWS Streamflow Forecast: 01a08717-6efc-7748-b71e-951a9bfc7452
Enabled: True
Next run: 2026-09-10 16:54:15.042617+00:00


### **What Happens Automatically?**

Once the task is enabled and HydroServer orchestration is running:

**GEOGLOWS API → Data Connection → ETL Task → HydroServer Datastream**

HydroServer's orchestration worker executes the ETL Task according to the configured schedule, so the Python notebook does not need to remain open.
